# 15 · Final model, external evaluation and 3D output (5-class)

1. **Final model.** The model selected in notebook 14 by the pre-declared rule is tuned once more on *all* development data, with the five outer folds as inner folds, then refitted on all development data.
2. **Freeze.** The model file and its SHA-256 are saved.
3. **External evaluation, once.** Only now are the 11 external boreholes' labels read. The depth-only reference is scored on the same samples, so the external result can be compared with it. Results are reported for all boreholes and for the interpolation and generalisation groups separately. In **smoke mode** the external boreholes are never touched: a stand-in set of development boreholes is used.
4. **3D prediction** on a 25 m × 1 m grid inside the Downtown polygon down to 20 m. It records the predicted class, the probability of every class, the normalised entropy (uncertainty), the reliability zone (distance to the nearest borehole: 1 ≤ 100 m, 2 ≤ 300 m, 3 > 300 m) and the extrapolation flags.

In [ ]:
import os, sys, json
from pathlib import Path

# 5-class soil model: this line selects config/project_5class.toml (keep it).
os.environ["SIEMREAP_CONFIG"] = "project_5class.toml"
# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner(), "|", cfg["_config_name"])

import joblib
from siemreap import nested_cv as NCV, metrics as M, grid3d, geo, folds as F
from siemreap.models import DepthPrior
from siemreap import data as D, plots as PL
CLS = D.class_info(cfg)["classes"]
NAMES = [c["name"] for c in CLS]
COLOURS = PL.class_colours(CLS)
K = len(NAMES)
AGG = config.stage_dir(cfg, "05_final", "aggregate")
RES = config.stage_dir(cfg, "05_final", "restricted")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
external = pd.read_csv(config.processed_dir(cfg) / "external_holdout.csv").set_index("BH_Code")
selected = json.loads((config.stage_dir(cfg, "04_results", "aggregate") / "selected_model.json").read_text())
print(json.dumps(selected, indent=1))

## 1. Final model

In [ ]:
sel = selected["selected"]
OVERRIDE = None        # a documented override must be reported, e.g. {"scenario": "A", "model": "RF", "block_m": ..., "reason": "..."}
if OVERRIDE:
    sel = OVERRIDE
final = NCV.fit_final(mt, cfg, sel["scenario"], sel["model"], int(sel["block_m"]), MODE)
print({k: final[k] for k in ("scenario", "model_name", "block_m", "params", "cv_macro_f1", "n_train_boreholes", "n_train_cells")})

## 2. Freeze

In [ ]:
model_path = RES / f"final_model_{sel['scenario']}_{sel['model']}_{sel['block_m']}m.joblib"
joblib.dump(final["model"], model_path, compress=3)
model_sha = config.sha256_file(model_path)
frozen = {"scenario": sel["scenario"], "model": sel["model"], "block_m": int(sel["block_m"]), "params": final["params"],
          "classes": NAMES, "cv_macro_f1_all_development": final["cv_macro_f1"],
          "n_train_boreholes": final["n_train_boreholes"], "n_train_samples": final["n_train_cells"],
          "top_elevation_range_m": final["top_elevation_range"], "model_file": model_path.name, "model_sha256": model_sha,
          "external_lock_sha256": json.loads((ROOT / "config" / "external_holdout.lock.json").read_text())["sha256"],
          "override": OVERRIDE, "mode": MODE, "environment": config.environment_record()}
(AGG / "final_model.json").write_text(json.dumps(frozen, indent=1, default=str))
final["trials"].to_csv(AGG / "final_model_optuna_trials.csv", index=False)
print("frozen:", model_path.name, model_sha[:16])

## 3. External evaluation (once, after freezing)

In [ ]:
ALLOW_REEVALUATION = False      # set True only with a documented reason; report it in the thesis
record_path = AGG / "external_evaluation.json"
if MODE == "smoke":
    rng = np.random.default_rng(0)
    dev_codes = mt[mt.in_A].BH_Code.drop_duplicates().sort_values().values
    stand_in = rng.choice(dev_codes, 8, replace=False)
    eval_cells = mt[mt.BH_Code.isin(stand_in)].assign(group="smoke_stand_in")
    label = "SMOKE STAND-IN (development boreholes; not an external evaluation)"
else:
    if record_path.exists():
        prev = json.loads(record_path.read_text())
        if prev["model_sha256"] != model_sha and not ALLOW_REEVALUATION:
            raise RuntimeError("The external set was already evaluated with a different frozen model; set "
                               "ALLOW_REEVALUATION only with a documented reason.")
    eval_cells = mt[mt.set == "external"].assign(group=lambda d: d.BH_Code.map(external.group))
    label = "EXTERNAL HOLDOUT"
train = mt[mt.in_A]
clay = D.clay_classes(cfg)
ref = DepthPrior(bottom_m=cfg["depth"]["bottom_m"], n_classes=K).fit(train.depth_top_m.values, train.y.values)
outs = {}
for name, P in ((sel["model"], final["model"].predict_proba(eval_cells[["UTM_Easting_m", "UTM_Northing_m", "Top_Elevation_m"]].values)),
                ("DEPTH_PRIOR", ref.predict_proba(eval_cells.depth_top_m.values))):
    yp = P.argmax(axis=1)
    outs[name] = eval_cells.assign(**{f"p_{c}": P[:, c] for c in range(K)}, y_pred=yp,
                                   y_pred_bin=np.isin(yp, clay).astype(int), p_clay=P[:, clay].sum(axis=1))
rows = []
for name, ev in outs.items():
    for g, d in [("all", ev), *ev.groupby("group")]:
        rows.append({"model": name, "group": g, **M.metric_row_mc(d.y, d[[f"p_{c}" for c in range(K)]].values, d.y_pred,
                                                                  d.y_bin, d.y_pred_bin)})
ext = pd.DataFrame(rows)
print(label)
display(ext[["model", "group", "n", "accuracy", "macro_f1", "kappa", "balanced_accuracy", "log_loss", "bin_accuracy", "bin_macro_f1"]].round(3))
ev = outs[sel["model"]]
ev["dist_nearest_train_m"] = geo.nearest_distance(ev[["UTM_Easting_m", "UTM_Northing_m"]].values, final["train_bh_xy"])
ev.to_csv(RES / "external_predictions.csv", index=False)
record_path.write_text(json.dumps({"label": label, "model_sha256": model_sha, "metrics": ext.to_dict("records"),
                                   "mode": MODE}, indent=1, default=str))
from siemreap import plots
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), dpi=150)
plots.confusion_plot(M.confusion_table(ev, K), NAMES, title=f"{label}: confusion (% recall)", ax=axes[0])
plots.confusion_plot(M.confusion_table(outs["DEPTH_PRIOR"], K), NAMES, title="Depth-only reference, same samples", ax=axes[1])
fig.tight_layout(); fig.savefig(AGG / "fig_external_confusion.png", bbox_inches="tight")
for k in ("scenario", "block_m", "purge_m", "model"):
    ev[k] = {"scenario": sel["scenario"], "block_m": sel["block_m"], "purge_m": 0.0, "model": sel["model"]}[k]
display(M.borehole_table(ev.assign(outer_fold=0))[["BH_Code", "n", "classes", "accuracy", "bin_accuracy", "dist_nearest_train_m"]].round(3))

## 4. 3D prediction for ArcGIS Pro

**Validated distance.** This is the 95th percentile of the test-to-nearest-training distances evaluated in the outer CV. Columns farther from a borehole than this are flagged `extrap_horizontal`. The reliability zones split the grid at 100 m and 300 m from the nearest borehole.

In [ ]:
preds = pd.read_csv(config.stage_dir(cfg, "03_nested_cv", "restricted") / "outer_predictions.csv")
pp = preds[(preds.block_m == sel["block_m"]) & (preds.purge_m == 0)]
validated_m = float(pp.drop_duplicates(["BH_Code", "model"]).dist_nearest_train_m.quantile(0.95))
print(f"validated horizontal distance (P95 of outer-CV test-to-train distances): {validated_m:.0f} m")
gcfg = {**cfg, "grid3d": {**cfg["grid3d"], "spacing_m": cfg["smoke"]["grid_spacing_m"] if MODE == "smoke" else cfg["grid3d"]["spacing_m"]}}
cols = grid3d.columns(gcfg)
grid = grid3d.cells(cols, gcfg)
grid = grid3d.predict_mc(final["model"], grid, cfg["grid3d"]["chunk_rows"], clay)
grid = grid3d.add_flags(grid, cols, final["train_bh_xy"], validated_m, final["top_elevation_range"], cfg["grid3d"]["zone_edges_m"])
print(f"{len(cols):,} columns x {grid.layer.nunique()} layers = {len(grid):,} cells")
share = {"horizontal_extrapolation": float(grid.extrap_horizontal.mean()), "vertical_extrapolation": float(grid.extrap_vertical.mean()),
         "reliability_zone_share": grid.reliability_zone.value_counts(normalize=True).sort_index().round(4).to_dict(),
         "predicted_class_share": {NAMES[c]: round(float(v), 4) for c, v in grid.soil_class.value_counts(normalize=True).sort_index().items()}}
print(json.dumps(share, indent=1))

In [ ]:
csv_path = RES / "siemreap_3d_points_5class.csv"
nc_path = RES / "siemreap_3d_voxels_5class.nc"
grid3d.to_points_csv(grid, csv_path)
try:
    vox = grid3d.to_voxel_netcdf(grid, cols, gcfg, nc_path)
except ImportError as e:
    vox = {"skipped": str(e)}
    print("netCDF export skipped (install xarray and netCDF4 to write it):", e)
summary3d = {"columns": int(len(cols)), "cells": int(len(grid)), "grid_spacing_m": gcfg["grid3d"]["spacing_m"],
             "layer_m": cfg["depth"]["cell_m"], "validated_distance_m": validated_m, **share, "voxel": vox,
             "classes": NAMES, "model_sha256": model_sha, "mode": MODE}
(AGG / "prediction_3d_summary.json").write_text(json.dumps(summary3d, indent=1, default=str))
(AGG / "class_legend.json").write_text(json.dumps({i: n for i, n in enumerate(NAMES)}, indent=1))

In [ ]:
# Quick look: predicted class at three depths (restricted figure: Downtown extent)
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), dpi=150, sharey=True)
for ax, d in zip(axes, [2.0, 8.0, 15.0]):
    g = grid[np.isclose(grid.depth_top_m, d)]
    ax.scatter(g.UTM_Easting_m, g.UTM_Northing_m, c=g.soil_class, cmap=ListedColormap(COLOURS), vmin=-0.5, vmax=K - 0.5,
               s=1 if MODE == "full" else 6, marker="s")
    h = g[g.reliability_zone == 3]
    ax.scatter(h.UTM_Easting_m, h.UTM_Northing_m, s=0.2, color="k", alpha=0.12)
    ax.set_title(f"Predicted class, {d:g}-{d + cfg['depth']['cell_m']:g} m (dots: > 300 m from a borehole)", fontsize=8.5, loc="left")
    ax.set_aspect("equal"); ax.ticklabel_format(useOffset=False, style="plain")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COLOURS]
fig.legend(handles, NAMES, loc="lower center", ncol=K, frameon=False, fontsize=8)
fig.savefig(RES / "fig_3d_depth_slices.png", bbox_inches="tight")

### Loading the outputs in ArcGIS Pro

* **3D points:** run *XY Table To Point* on `siemreap_3d_points_5class.csv` with X = `X`, Y = `Y`, Z = `Z` and coordinate system EPSG:32648. Symbolise by `soil_class` using the legend in `class_legend.json`, and use `reliability_zone`, `p_max` and `entropy` to show where the map can be trusted.
* **Voxel layer:** in a local scene, use *Add Voxel Layer* with `siemreap_3d_voxels_5class.nc` (variables `soil_class`, `p_max`, `p_clay`, `entropy`, `reliability_zone`, `extrap_horizontal`, `extrap_vertical`).
* Both files are in `outputs_5class/05_final/restricted/`, which is git-ignored.